# Kalshi Market Data Demo

Demonstrates fetching open markets and orderbooks using the `kalshi_client` module.

## Setup

1. `pip install kalshi-python python-dotenv pandas`
2. Copy `.env.example` to `.env` and fill in your API credentials
3. Set `KALSHI_ENV=demo` for the demo exchange or `KALSHI_ENV=production` for live

In [ ]:
# Install dependencies (uncomment in Colab)
# !pip install kalshi-python python-dotenv pandas

In [ ]:
from kalshi_client import KalshiMarketClient

# Reads KALSHI_ENV, KALSHI_API_KEY_ID, KALSHI_PRIVATE_KEY_PATH from .env
client = KalshiMarketClient()
print(f"Connected to Kalshi ({client.env})")

## 1. Fetch open markets (first page)

In [ ]:
df = client.get_open_markets(limit=20, max_pages=1)
print(f"{len(df)} markets returned")
df[["ticker", "title", "status", "yes_ask", "no_ask"]].head(10)

## 2. Filter by event ticker

Pass `event_ticker` to narrow results to a specific event.

In [ ]:
# Example: filter by a specific event ticker
# Replace with a real event ticker from the DataFrame above
sample_event = df["event_ticker"].dropna().iloc[0] if "event_ticker" in df.columns and not df["event_ticker"].dropna().empty else None
if sample_event:
    event_df = client.get_open_markets(event_ticker=sample_event)
    print(f"Markets for event '{sample_event}': {len(event_df)}")
    event_df[["ticker", "title"]].head()

## 3. Filter by series ticker

In [ ]:
# Example: filter by series ticker
# Common series: "KXBTC" (Bitcoin), "KXINX" (S&P 500), etc.
# btc_df = client.get_open_markets(series_ticker="KXBTC")
# btc_df[["ticker", "title"]].head()

## 4. Search markets by keyword

In [ ]:
# Case-insensitive title search across open markets
results = client.search_markets("Bitcoin", limit=200)
print(f"Found {len(results)} markets matching 'Bitcoin'")
results[["ticker", "title"]].head(10)

## 5. Filter by close timestamp

Use `min_close_ts` / `max_close_ts` (Unix timestamps) to find markets
closing within a time window.

In [ ]:
import time

# Markets closing in the next 24 hours
now = int(time.time())
soon_df = client.get_open_markets(
    min_close_ts=now,
    max_close_ts=now + 86400,
    max_pages=1,
)
print(f"{len(soon_df)} markets closing in the next 24h")
soon_df[["ticker", "title"]].head(10)

## 6. Orderbook (depth=10 default)

In [ ]:
if not df.empty:
    sample_ticker = df["ticker"].iloc[0]
    ob = client.get_orderbook(sample_ticker)
    print(f"Orderbook for {ob['ticker']}")
    print("\nYES bids:")
    display(ob["yes"])
    print("\nNO bids:")
    display(ob["no"])

## 7. Single market detail

In [ ]:
if not df.empty:
    detail = client.get_market(df["ticker"].iloc[0])
    detail